# Retail Sales Data Analysis and Visualization Using Python

**Student:** Kalyanam Shanthi Priya  
**Email:** kalyanamshanthipriya021@gmail.com  
**Degree/Branch:** B.Tech Information Technology  
**Institution:** Mohan Babu University, Tirupati, Andhra Pradesh, India  
**Internship:** AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026

## Project objective
Analyze historical retail transactions to understand sales value, product performance, monthly patterns, country-level performance, customer activity, and data quality. Use computed evidence to propose practical business recommendations.

## Important reproducibility note
This notebook does not include the original dataset. Download the UCI Online Retail file, convert it to CSV, and save it as `data/Online Retail.csv` relative to this notebook. The notebook computes findings from your local data; no results are hardcoded.

**Dataset source:** https://archive.ics.uci.edu/dataset/352/online+retail  
**Dataset citation:** Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33. License: CC BY 4.0.

## 1. Business problem statement
Retail businesses need to understand which products and markets contribute most to transaction value, how sales change over time, and how returns and data-quality issues affect reporting. This project demonstrates a reproducible analysis of historical online retail transactions to support better inventory, merchandising, and customer-retention decisions.

## 2. Research questions
1. What are the positive transaction value and net transaction value after negative-quantity records are included?
2. Which products rank highest by net transaction value and quantity sold?
3. How does transaction value vary month by month?
4. Which countries contribute most to transaction value?
5. What can be learned about customer purchase activity from records with a customer ID?
6. How many records are missing, duplicated, cancelled, returned, or otherwise unsuitable for particular analyses?

## 3. Dataset description
The UCI Online Retail dataset describes transactions for a UK-based online retailer between December 2010 and December 2011. Important columns include `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country`. Cancellation invoices may start with `C`; negative quantities can represent returns. The dataset contains no profit margin or discount column, so profit and discount effects are not analyzed.

## 4. Workflow
Load data → inspect data quality → clean types and records → engineer transaction value → summarize KPIs → visualize patterns → interpret computed results → document recommendations.

## 5. Install and import libraries
Run `pip install -r requirements.txt` in the project environment before running the notebook. This project uses pandas and NumPy for data handling and Matplotlib/Seaborn for visualizations.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "Online Retail.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Notebook working directory:", PROJECT_DIR)
print("Expected CSV path:", DATA_PATH)

## 6. Load the dataset
Download `Online Retail.xlsx` from the official UCI page. Open it in Excel and choose **Save As → CSV UTF-8 (Comma delimited)**. Save the result as `Online Retail.csv` inside the project's `data` folder. If the file is missing, the next cell prints the exact expected location and stops with a helpful error.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset CSV not found. Download Online Retail.xlsx from "
        "https://archive.ics.uci.edu/dataset/352/online+retail, save it as "
        "'Online Retail.csv', and place it at: " + str(DATA_PATH)
    )

df_raw = pd.read_csv(DATA_PATH, encoding="utf-8", low_memory=False)
print(f"Loaded {len(df_raw):,} rows and {df_raw.shape[1]:,} columns.")
display(df_raw.head())

## 7. Initial inspection
Use `head()`, `info()`, `describe()`, and shape checks to understand the dataset before changing it. Review the actual column names because CSV exports can sometimes alter formatting.

In [ ]:
print("Shape:", df_raw.shape)
print("\nColumn names:", df_raw.columns.tolist())
display(df_raw.head(10))
print("\nData types and non-null counts:")
df_raw.info()
print("\nDescriptive statistics:")
display(df_raw.describe(include="all").T)
print("\nMissing values by column:")
display(df_raw.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nExact duplicate rows:", int(df_raw.duplicated().sum()))

## 8. Standardize columns and check data quality
Column names are normalized to expected names. Required columns are checked before analysis. Date and numeric fields are converted with invalid values becoming missing so they can be counted and handled transparently.

In [ ]:
# Normalize common exported column labels while preserving a clear expected schema.
rename_map = {
    "InvoiceNo": "InvoiceNo", "Invoice": "InvoiceNo", "Invoice No": "InvoiceNo",
    "StockCode": "StockCode", "Stock Code": "StockCode",
    "Description": "Description", "Quantity": "Quantity",
    "InvoiceDate": "InvoiceDate", "Invoice Date": "InvoiceDate",
    "UnitPrice": "UnitPrice", "Unit Price": "UnitPrice",
    "CustomerID": "CustomerID", "Customer ID": "CustomerID",
    "Country": "Country"
}
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]
# Match known names case-insensitively.
lookup = {str(c).strip().lower().replace("_", "").replace(" ", ""): c for c in df.columns}
canonical = {
    "invoiceno":"InvoiceNo", "invoice":"InvoiceNo", "stockcode":"StockCode",
    "description":"Description", "quantity":"Quantity", "invoicedate":"InvoiceDate",
    "unitprice":"UnitPrice", "customerid":"CustomerID", "country":"Country"
}
df = df.rename(columns={original: canonical[key] for key, original in lookup.items() if key in canonical})

required = ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "Country"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}. Check the CSV export and column names.")

for col in ["Quantity", "UnitPrice"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce", dayfirst=True)
if "CustomerID" in df.columns:
    df["CustomerID"] = df["CustomerID"].astype("string").str.replace(r"\.0$", "", regex=True)

quality_before = {
    "rows_loaded": len(df),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "missing_invoice_date": int(df["InvoiceDate"].isna().sum()),
    "missing_quantity": int(df["Quantity"].isna().sum()),
    "missing_unit_price": int(df["UnitPrice"].isna().sum()),
    "missing_description": int(df["Description"].isna().sum()),
    "missing_country": int(df["Country"].isna().sum()),
}
display(pd.Series(quality_before, name="count").to_frame())

## 9. Cleaning strategy
Exact duplicate rows are removed. Records without a usable date, quantity, or unit price cannot contribute reliably to time or value analysis and are excluded from the analytical table. Missing customer IDs are retained for overall sales analysis but excluded from customer-level analysis. Negative quantities are **not automatically deleted** because they can represent returns. Rows with non-positive unit prices are flagged and excluded from the primary sales-value analysis; inspect them separately if needed.

In [ ]:
rows_before = len(df)
df = df.drop_duplicates().copy()
df = df.dropna(subset=["InvoiceDate", "Quantity", "UnitPrice"]).copy()
df["Description"] = df["Description"].astype("string").str.strip()
df["Country"] = df["Country"].astype("string").str.strip()
df["InvoiceNo"] = df["InvoiceNo"].astype("string").str.strip()
df["StockCode"] = df["StockCode"].astype("string").str.strip()

# Retain all usable transaction lines, including negative quantities, for net-value calculations.
df["LineValue"] = df["Quantity"] * df["UnitPrice"]
df["IsCancellationInvoice"] = df["InvoiceNo"].str.upper().str.startswith("C", na=False)
df["IsReturnOrNegativeQuantity"] = df["Quantity"] < 0
df["IsPositivePrice"] = df["UnitPrice"] > 0
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["YearMonthDate"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
df["Year"] = df["InvoiceDate"].dt.year
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()

analysis_df = df.loc[df["IsPositivePrice"]].copy()
# Positive sales exclude negative-quantity records; net transaction value includes their negative values.
positive_sales_df = analysis_df.loc[analysis_df["Quantity"] > 0].copy()
negative_quantity_df = analysis_df.loc[analysis_df["Quantity"] < 0].copy()

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after duplicate and required-field cleaning: {len(df):,}")
print(f"Rows with positive unit price used for primary analysis: {len(analysis_df):,}")
print(f"Rows with positive quantity: {len(positive_sales_df):,}")
print(f"Rows with negative quantity (potential returns): {len(negative_quantity_df):,}")
print(f"Rows with zero quantity: {(analysis_df['Quantity'] == 0).sum():,}")
print(f"Rows with non-positive unit price excluded from primary analysis: {(df['UnitPrice'] <= 0).sum():,}")

## 10. Data-quality summary and outlier inspection
Outliers are inspected rather than removed automatically: high-value wholesale orders can be legitimate. The interquartile range (IQR) is used only to flag unusual positive line values for review. It is not used as an automatic deletion rule.

In [ ]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Original rows", "Exact duplicate rows in original data",
        "Rows remaining after duplicate removal and required-field checks",
        "Missing customer IDs after cleaning", "Cancellation invoice rows",
        "Negative quantity rows", "Non-positive price rows"
    ],
    "Count": [
        len(df_raw), int(df_raw.duplicated().sum()), len(df),
        int(df["CustomerID"].isna().sum()) if "CustomerID" in df.columns else len(df),
        int(df["IsCancellationInvoice"].sum()), int((df["Quantity"] < 0).sum()),
        int((df["UnitPrice"] <= 0).sum())
    ]
})
display(quality_summary)
positive_line_values = positive_sales_df["LineValue"].dropna()
if len(positive_line_values):
    q1, q3 = positive_line_values.quantile([0.25, 0.75])
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    outlier_count = int((positive_line_values > upper_fence).sum())
    print(f"IQR upper fence for positive line value: £{upper_fence:,.2f}")
    print(f"Positive line-value records above this fence (flagged, not removed): {outlier_count:,}")
else:
    print("No positive line values available for outlier inspection.")

## 11. Key performance indicators (KPIs)
The terms are defined carefully: **positive transaction value** sums positive-quantity line values; **net transaction value** sums line values including negative quantities, after filtering non-positive unit prices. These are transaction-value measures, not accounting profit and not necessarily final recognized revenue.

In [ ]:
kpis = {
    "Rows analyzed (positive unit price)": len(analysis_df),
    "Unique invoices": int(analysis_df["InvoiceNo"].nunique()),
    "Unique products": int(analysis_df["StockCode"].nunique()),
    "Unique countries": int(analysis_df["Country"].nunique()),
    "Positive transaction value (£)": positive_sales_df["LineValue"].sum(),
    "Net transaction value (£)": analysis_df["LineValue"].sum(),
    "Units in positive-quantity lines": positive_sales_df["Quantity"].sum(),
    "Potential return value (£)": negative_quantity_df["LineValue"].sum(),
}
if "CustomerID" in analysis_df.columns:
    kpis["Identified customers"] = int(analysis_df["CustomerID"].dropna().nunique())
display(pd.DataFrame({"KPI": list(kpis.keys()), "Computed value": list(kpis.values())}))

## 12. Monthly sales trends
Monthly summaries show how transaction value changes across the dataset period. The line chart is generated from the loaded file, and the monthly summary is saved to `outputs/monthly_sales_summary.csv`.

In [ ]:
monthly = analysis_df.groupby("YearMonthDate", as_index=False).agg(
    net_transaction_value=("LineValue", "sum"),
    line_items=("InvoiceNo", "size"),
    invoices=("InvoiceNo", "nunique")
)
monthly["positive_transaction_value"] = positive_sales_df.groupby("YearMonthDate")["LineValue"].sum().reindex(monthly["YearMonthDate"]).to_numpy()
monthly["positive_transaction_value"] = monthly["positive_transaction_value"].fillna(0)
display(monthly)
monthly.to_csv(OUTPUT_DIR / "monthly_sales_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["YearMonthDate"], monthly["net_transaction_value"], marker="o", label="Net transaction value")
ax.plot(monthly["YearMonthDate"], monthly["positive_transaction_value"], marker=".", alpha=0.8, label="Positive transaction value")
ax.set_title("Monthly Transaction Value")
ax.set_xlabel("Month")
ax.set_ylabel("Value (£)")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "monthly_sales_trend.png", dpi=160, bbox_inches="tight")
plt.show()

## 13. Product performance
Product descriptions may be missing or inconsistent, so product code is retained as a stable grouping key. The tables rank products by net line value and units in positive-quantity records. Rankings are calculated from the local data.

In [ ]:
product_net = analysis_df.groupby(["StockCode", "Description"], dropna=False, as_index=False).agg(
    net_transaction_value=("LineValue", "sum"),
    net_quantity=("Quantity", "sum"),
    line_items=("InvoiceNo", "size")
).sort_values("net_transaction_value", ascending=False)
product_units = positive_sales_df.groupby(["StockCode", "Description"], dropna=False, as_index=False).agg(
    units_sold=("Quantity", "sum"),
    positive_transaction_value=("LineValue", "sum")
).sort_values("units_sold", ascending=False)

print("Top 10 products by net transaction value:")
display(product_net.head(10))
print("Top 10 products by positive units sold:")
display(product_units.head(10))
product_net.head(20).to_csv(OUTPUT_DIR / "top_products_by_net_value.csv", index=False)
product_units.head(20).to_csv(OUTPUT_DIR / "top_products_by_units.csv", index=False)

plot_products = product_net.head(10).sort_values("net_transaction_value")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_products["Description"].fillna(plot_products["StockCode"]).astype(str), plot_products["net_transaction_value"])
ax.set_title("Top 10 Products by Net Transaction Value")
ax.set_xlabel("Net transaction value (£)")
ax.set_ylabel("Product")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "top_products_by_net_value.png", dpi=160, bbox_inches="tight")
plt.show()

## 14. Country-level performance
The dataset contains country rather than a formal sales region or customer segment. This section compares transaction value by country. Small markets and missing country labels should be interpreted with care.

In [ ]:
country_sales = analysis_df.groupby("Country", dropna=False, as_index=False).agg(
    net_transaction_value=("LineValue", "sum"),
    invoices=("InvoiceNo", "nunique"),
    line_items=("InvoiceNo", "size")
).sort_values("net_transaction_value", ascending=False)
display(country_sales.head(15))
country_sales.to_csv(OUTPUT_DIR / "country_sales_summary.csv", index=False)

top_countries = country_sales.head(10).sort_values("net_transaction_value")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_countries["Country"].astype(str), top_countries["net_transaction_value"])
ax.set_title("Top 10 Countries by Net Transaction Value")
ax.set_xlabel("Net transaction value (£)")
ax.set_ylabel("Country")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "country_sales.png", dpi=160, bbox_inches="tight")
plt.show()

## 15. Customer purchasing activity
Customer-level analysis is limited to rows with a non-missing customer ID. The table reports number of invoices and transaction value; it should not be interpreted as customer lifetime value because the dataset covers a limited historical period and may not include all customer activity.

In [ ]:
if "CustomerID" in analysis_df.columns:
    customer_df = analysis_df.dropna(subset=["CustomerID"]).copy()
    customer_summary = customer_df.groupby("CustomerID", as_index=False).agg(
        invoices=("InvoiceNo", "nunique"),
        net_transaction_value=("LineValue", "sum"),
        line_items=("InvoiceNo", "size"),
        last_observed_purchase=("InvoiceDate", "max")
    ).sort_values("net_transaction_value", ascending=False)
    print("Top 10 identified customers by net transaction value:")
    display(customer_summary.head(10))
    customer_summary.to_csv(OUTPUT_DIR / "customer_summary.csv", index=False)
else:
    print("CustomerID column is not available; customer analysis skipped.")

## 16. Correlation analysis
Correlation is computed only for numeric transaction variables. Correlation is descriptive and does not establish causation. Invoice-level quantity and line value are not independent business measures, so interpret the result in context.

In [ ]:
numeric_cols = [c for c in ["Quantity", "UnitPrice", "LineValue"] if c in analysis_df.columns]
corr = analysis_df[numeric_cols].corr(numeric_only=True)
display(corr)
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Among Numeric Transaction Variables")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "numeric_correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 17. Optional machine-learning extension: monthly forecasting
Forecasting is optional and is not included as a mandatory dependency. A credible forecast requires enough ordered monthly observations and careful temporal validation. This starter project focuses on descriptive analysis first; it does not report forecast accuracy without a separately implemented and evaluated model. Future work could compare a seasonal-naive baseline with a time-series model using a chronological holdout period.

## 18. Automatically generated findings
The following summary uses computed outputs from the dataset loaded in this notebook. Read the output after executing the notebook; do not copy unverified values into a submission report.

In [ ]:
def gbp(value):
    return f"£{value:,.2f}"

best_product = product_net.iloc[0] if len(product_net) else None
best_country = country_sales.iloc[0] if len(country_sales) else None
peak_month = monthly.loc[monthly["net_transaction_value"].idxmax()] if len(monthly) else None

finding_lines = [
    f"- Rows included in primary transaction analysis: **{len(analysis_df):,}**.",
    f"- Positive transaction value: **{gbp(positive_sales_df['LineValue'].sum())}**.",
    f"- Net transaction value including negative quantities: **{gbp(analysis_df['LineValue'].sum())}**.",
    f"- Distinct invoices: **{analysis_df['InvoiceNo'].nunique():,}**; distinct products: **{analysis_df['StockCode'].nunique():,}**."
]
if best_product is not None:
    finding_lines.append(f"- Highest-ranked product by net transaction value: **{best_product['Description']}** (code {best_product['StockCode']}), **{gbp(best_product['net_transaction_value'])}**.")
if best_country is not None:
    finding_lines.append(f"- Highest-ranked country by net transaction value: **{best_country['Country']}**, **{gbp(best_country['net_transaction_value'])}**.")
if peak_month is not None:
    finding_lines.append(f"- Month with the highest net transaction value in the analyzed data: **{peak_month['YearMonthDate'].strftime('%Y-%m')}**, **{gbp(peak_month['net_transaction_value'])}**.")

display(Markdown("\n".join(finding_lines)))
(OUTPUT_DIR / "computed_findings.md").write_text("\n".join(finding_lines) + "\n", encoding="utf-8")

## 19. Business recommendations
Use the computed rankings and charts to support recommendations, rather than treating recommendations as predetermined facts.

- **Inventory planning:** Review products that rank highly by both value and quantity before setting replenishment priorities.
- **Merchandising:** Compare product-value rankings with unit rankings to identify high-volume and high-value products.
- **Seasonal planning:** Use monthly patterns to plan staffing, campaigns, and stock levels, while recognizing that this dataset covers only about one year.
- **Market strategy:** Compare country-level transaction value and invoice counts; investigate shipping costs and profitability separately before allocating budgets.
- **Returns monitoring:** Review negative-quantity and cancellation records to understand how returns affect net transaction value.
- **Customer retention:** For identified customers, investigate repeat-purchase frequency and value while acknowledging missing customer IDs.

These are general decision options. Select the recommendations that are justified by your computed findings.

## 20. Limitations
- The data is historical and does not represent current retail performance.
- Profit, cost, discount, shipping, and product-category fields are unavailable.
- Missing customer IDs limit customer-level conclusions.
- Negative quantities and cancellation invoices require careful interpretation.
- Outliers are flagged for review rather than automatically removed.
- A one-year time span is insufficient for robust claims about long-term seasonality.
- This project is descriptive analytics, not an audited financial statement or causal study.

## 21. Conclusion
This notebook provides a repeatable workflow for inspecting and cleaning retail transactions, calculating transparent transaction-value metrics, visualizing product/month/country patterns, and producing data-derived findings. Execute all cells from top to bottom after placing the dataset in the expected location. Use the resulting charts and `outputs/computed_findings.md` to update the final project report.

## 22. References
1. Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33
2. UCI dataset page: https://archive.ics.uci.edu/dataset/352/online+retail
3. pandas documentation: https://pandas.pydata.org/docs/
4. Matplotlib documentation: https://matplotlib.org/stable/
5. Seaborn documentation: https://seaborn.pydata.org/